**Window Function**
 
PySpark Window functions are used to calculate results, such as the rank, row number, etc.

row_number(): Returns a sequential number starting from 1 within a window partition.  

rank(): Returns the rank of rows within a window partition, with gaps.  

dense_rank(): Returns the rank of rows within a window partition without any gaps. Whereas rank() returns rank with gaps.  

lag(): In PySpark, the lag() function retrieves the value of a column from a preceding row within the same window.

lead(): Function in PySpark retrieves the value of a column from a succeeding row within the same window.

In [0]:
data = [
    (1, "Alice", "Sales", 6000),
    (2, "Bob", "Sales", 4500),
    (3, "Charlie", "Sales", 6000),
    (4, "David", "Sales", 3000),
    (5, "Eve", "HR", 7500),
    (6, "Frank", "HR", 4500),
    (7, "Grace", "HR", 8000),
    (8, "Hank", "HR", 8000)
]

columns = ["id", "name", "dept", "salary"]
df = spark.createDataFrame(data, columns)
df.show()

+---+-------+-----+------+
| id|   name| dept|salary|
+---+-------+-----+------+
|  1|  Alice|Sales|  6000|
|  2|    Bob|Sales|  4500|
|  3|Charlie|Sales|  6000|
|  4|  David|Sales|  3000|
|  5|    Eve|   HR|  7500|
|  6|  Frank|   HR|  4500|
|  7|  Grace|   HR|  8000|
|  8|   Hank|   HR|  8000|
+---+-------+-----+------+



In [0]:
df.createOrReplaceTempView("employee")

In [0]:
%sql
select *, row_number() over(partition by dept order by salary desc)  rn
from employee

id,name,dept,salary,rn
7,Grace,HR,8000,1
8,Hank,HR,8000,2
5,Eve,HR,7500,3
6,Frank,HR,4500,4
1,Alice,Sales,6000,1
3,Charlie,Sales,6000,2
2,Bob,Sales,4500,3
4,David,Sales,3000,4


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import * 

w = Window.partitionBy("dept").orderBy(col('salary').desc())
df.withColumn('rn', row_number().over(w)).show()


+---+-------+-----+------+---+
| id|   name| dept|salary| rn|
+---+-------+-----+------+---+
|  7|  Grace|   HR|  8000|  1|
|  8|   Hank|   HR|  8000|  2|
|  5|    Eve|   HR|  7500|  3|
|  6|  Frank|   HR|  4500|  4|
|  1|  Alice|Sales|  6000|  1|
|  3|Charlie|Sales|  6000|  2|
|  2|    Bob|Sales|  4500|  3|
|  4|  David|Sales|  3000|  4|
+---+-------+-----+------+---+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import * 

w = Window.partitionBy("dept").orderBy(col('salary').desc())
df.withColumn('rank', rank().over(w)).show()





+---+-------+-----+------+----+
| id|   name| dept|salary|rank|
+---+-------+-----+------+----+
|  7|  Grace|   HR|  8000|   1|
|  8|   Hank|   HR|  8000|   1|
|  5|    Eve|   HR|  7500|   3|
|  6|  Frank|   HR|  4500|   4|
|  1|  Alice|Sales|  6000|   1|
|  3|Charlie|Sales|  6000|   1|
|  2|    Bob|Sales|  4500|   3|
|  4|  David|Sales|  3000|   4|
+---+-------+-----+------+----+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import * 

w = Window.partitionBy("dept").orderBy(col('salary').desc())
df.withColumn('dn', dense_rank().over(w)).show()





+---+-------+-----+------+---+
| id|   name| dept|salary| dn|
+---+-------+-----+------+---+
|  7|  Grace|   HR|  8000|  1|
|  8|   Hank|   HR|  8000|  1|
|  5|    Eve|   HR|  7500|  2|
|  6|  Frank|   HR|  4500|  3|
|  1|  Alice|Sales|  6000|  1|
|  3|Charlie|Sales|  6000|  1|
|  2|    Bob|Sales|  4500|  2|
|  4|  David|Sales|  3000|  3|
+---+-------+-----+------+---+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import * 

w = Window.partitionBy("dept").orderBy(col('salary').desc())
df.withColumn('ps', lag("salary").over(w)).show()


+---+-------+-----+------+----+
| id|   name| dept|salary|  ps|
+---+-------+-----+------+----+
|  7|  Grace|   HR|  8000|NULL|
|  8|   Hank|   HR|  8000|8000|
|  5|    Eve|   HR|  7500|8000|
|  6|  Frank|   HR|  4500|7500|
|  1|  Alice|Sales|  6000|NULL|
|  3|Charlie|Sales|  6000|6000|
|  2|    Bob|Sales|  4500|6000|
|  4|  David|Sales|  3000|4500|
+---+-------+-----+------+----+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import * 

w = Window.partitionBy("dept").orderBy(col('salary').desc())
df.withColumn('ns', lead("salary").over(w)).show()

+---+-------+-----+------+----+
| id|   name| dept|salary|  ns|
+---+-------+-----+------+----+
|  7|  Grace|   HR|  8000|8000|
|  8|   Hank|   HR|  8000|7500|
|  5|    Eve|   HR|  7500|4500|
|  6|  Frank|   HR|  4500|NULL|
|  1|  Alice|Sales|  6000|6000|
|  3|Charlie|Sales|  6000|4500|
|  2|    Bob|Sales|  4500|3000|
|  4|  David|Sales|  3000|NULL|
+---+-------+-----+------+----+

